In [3]:
from mods import *
pd.set_option('display.max_columns', None)
warnings.filterwarnings("ignore", message="DataFrame.swapaxes is deprecated")
warnings.filterwarnings('ignore', category=FutureWarning)
# folder_path - Путь к папке с исходными необработанными данными
folder_raw_path = r"E:\Programming\python\SC_anomaly_classifier\Data\supercomputer_timeseries\train_w_areas_st_till_june.csv"

In [4]:
df = pd.read_pickle(r"E:\Programming\python\SC_anomaly_classifier\Data\Processed\prepared_df.pkl")
X = pd.read_pickle(r"E:\Programming\python\SC_anomaly_classifier\Data\Processed\X_raw.pkl")
y = pd.read_pickle(r"E:\Programming\python\SC_anomaly_classifier\Data\Processed\y.pkl")
X_enc = pd.read_pickle(r"E:\Programming\python\SC_anomaly_classifier\Data\Processed\X_enc_OHE_not_norm.pkl")
y_enc = pd.read_pickle(r"E:\Programming\python\SC_anomaly_classifier\Data\Processed\y_enc_OHE.pkl")
X_enc_q_norm = pd.read_pickle(r"E:\Programming\python\SC_anomaly_classifier\Data\Processed\X_enc_OHE_quantile_norm.pkl")
X_enc_log_std_norm = pd.read_pickle(r"E:\Programming\python\SC_anomaly_classifier\Data\Processed\X_enc_OHE_log_std_norm.pkl")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.metrics import classification_report, precision_recall_fscore_support
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputClassifier
from sklearn.utils import class_weight
import warnings
warnings.filterwarnings('ignore')

half_size = len(X_enc_log_std_norm) // 20
X = X_enc_log_std_norm.iloc[:half_size]
y = y_enc.iloc[:half_size]


# Преобразование one-hot в мультикласс
y_classes = y.idxmax(axis=1)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_classes)

# Разделение на train/validation (80/20)
X_train, X_val, y_train, y_val = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Проверка баланса классов
print("Распределение классов:")
print(pd.Series(y_train).value_counts(normalize=True))

# Функция для вычисления метрик с учетом дисбаланса
def evaluate_model(model, X, y, cv=5):
    # Получаем предсказания вероятностей и классов
    y_pred_proba = cross_val_predict(model, X, y, cv=cv, method='predict_proba')
    y_pred_class = np.argmax(y_pred_proba, axis=1)
    
    # Если y в one-hot формате, преобразуем в числовые метки
    if len(y.shape) > 1 and y.shape[1] > 1:
        y_true = np.argmax(y.values if isinstance(y, pd.DataFrame) else y, axis=1)
    else:
        y_true = y
    
    # Вычисляем метрики
    precision_per_class, recall_per_class, f1_per_class, support_per_class = \
        precision_recall_fscore_support(y_true, y_pred_class, average=None)
    
    # Вычисляем усредненные метрикиы
    macro_precision, macro_recall, macro_f1, _ = \
        precision_recall_fscore_support(y_true, y_pred_class, average='macro')
    
    # Формируем отчет
    class_report = classification_report(y_true, y_pred_class, output_dict=True)
    
    # Собираем результаты
    results = {
    
        'macro': {
            'precision': macro_precision,
            'recall': macro_recall,
            'f1': macro_f1
        },
        'per_class': {
            'precision': precision_per_class.tolist(),
            'recall': recall_per_class.tolist(),
            'f1': f1_per_class.tolist(),
            'support': support_per_class.tolist()
    }}
    
    return results

# Список моделей с оптимальными параметрами
models = [
    ('RandomForest', RandomForestClassifier(
        n_estimators=200,
        class_weight='balanced',
        max_depth=10,
        random_state=42
    )),
    ('XGBoost', XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        use_label_encoder=False,
        eval_metric='mlogloss',
        scale_pos_weight=len(y_train)/np.bincount(y_train)
    )),
    ('LightGBM', LGBMClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        class_weight='balanced'
    )),
    ('LogisticRegression', LogisticRegression(
            multi_class='multinomial',
            solver='lbfgs',
            max_iter=1000,
            class_weight='balanced'
        )),
    ('SVM',  SVC(
            probability=True,
            class_weight='balanced',
            kernel='rbf',
            C=1.0
        )),
    ('MLP', MLPClassifier(
            hidden_layer_sizes=(100,50),
            max_iter=1000,
            early_stopping=True
        ))
]

# Оценка моделей
results = []
for name, model in models:
    print(f"\nОценка модели: {name}")
    metrics = evaluate_model(model, X_train, y_train)
    print(metrics)
    results.append({'Model': name, **metrics})

# Сравнение моделей
results_df = pd.DataFrame(results).sort_values('F1', ascending=False)
print("\nСравнение моделей:")
print(results_df)

# Обучение лучшей модели
best_model_name = results_df.iloc[0]['Model']
best_model = [m for m in models if m[0] == best_model_name][0][1]
best_model.fit(X_train, y_train)

# Оценка на валидационной выборке
y_val_pred = best_model .predict(X_val)
print("\nЛучшая модель:", best_model_name)
print(classification_report(y_val, y_val_pred, target_names=label_encoder.classes_))



In [20]:
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import numpy as np

def custom_metrics(y_true, y_pred):
    """
    Вычисляет метрики для каждого класса (1,2,3) и их средние значения.
    Оптимизирует под минимальное количество ложных срабатываний (FP).
    """
    # Метрики для каждого класса
    metrics = {}
    for class_id in [1, 2, 3]:
        # Для каждого класса считаем бинарные метрики
        y_true_bin = (y_true == class_id).astype(int)
        y_pred_bin = (y_pred == class_id).astype(int)
        
        precision = precision_score(y_true_bin, y_pred_bin, zero_division=0)
        recall = recall_score(y_true_bin, y_pred_bin, zero_division=0)
        f1 = f1_score(y_true_bin, y_pred_bin, zero_division=0)
        
        metrics[f'class_{class_id}'] = {
            'precision': precision,
            'recall': recall,
            'f1': f1
        }
    
    # Средние метрики для классов 1,2,3
    avg_metrics = {
        'avg_precision': np.mean([metrics[f'class_{i}']['precision'] for i in [1, 2, 3]]),
        'avg_recall': np.mean([metrics[f'class_{i}']['recall'] for i in [1, 2, 3]]),
        'avg_f1': np.mean([metrics[f'class_{i}']['f1'] for i in [1, 2, 3]])
    }
    
    return {**metrics, **avg_metrics}

class FPOptimizedXGBoost(XGBClassifier):
    """
    Модифицированный XGBoost с оптимизацией под минимальное количество FP.
    """
    def __init__(self, fp_weight=2.0, **kwargs):
        super().__init__(**kwargs)
        self.fp_weight = fp_weight  # Вес для штрафа за ложные срабатывания
    
    def fit(self, X, y, **kwargs):
        # Добавляем веса классов с акцентом на уменьшение FP
        class_weights = {0: 1, 1: self.fp_weight, 2: self.fp_weight, 3: self.fp_weight}
        sample_weight = np.array([class_weights[c] for c in y])
        return super().fit(X, y, sample_weight=sample_weight, **kwargs)


if __name__ == "__main__":
    y = y_encoded
    half_size = len(X_enc_log_std_norm) // 20
    X = X_enc_log_std_norm.iloc[:half_size]
    # Обучаем модифицированный XGBoost
    model = FPOptimizedXGBoost(fp_weight=3.0, 
                              eval_metric='logloss',
                              use_label_encoder=False)
    model.fit(X, y)
    
    # Предсказания
    y_pred = model.predict(X)
    
    # Вычисляем кастомные метрики
    metrics = custom_metrics(y, y_pred)
    print("Метрики по классам и средние:")
    for k, v in metrics.items():
        print(f"{k}: {v}")

Метрики по классам и средние:
class_1: {'precision': 0.5992353145637818, 'recall': 0.8521997034107761, 'f1': 0.7036734693877551}
class_2: {'precision': 0.7672955974842768, 'recall': 0.4485294117647059, 'f1': 0.5661252900232019}
class_3: {'precision': 0.723744292237443, 'recall': 0.698237885462555, 'f1': 0.7107623318385651}
avg_precision: 0.6967584014285005
avg_recall: 0.6663223335460123
avg_f1: 0.6601870304165073


Метрики по классам и средние:  
class_1: {'precision': 0.5992353145637818, 'recall': 0.8521997034107761, 'f1': 0.7036734693877551}  
class_2: {'precision': 0.7672955974842768, 'recall': 0.4485294117647059, 'f1': 0.5661252900232019}  
class_3: {'precision': 0.723744292237443, 'recall': 0.698237885462555, 'f1': 0.7107623318385651}  
avg_precision: 0.6967584014285005  
avg_recall: 0.6663223335460123  
avg_f1: 0.6601870304165073  

In [34]:
len(X_enc_log_std_norm)

1290728

In [ ]:
# Случайный лес с бустингом (Stacking)
from sklearn.ensemble import StackingClassifier

base_models = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('xgb', XGBClassifier(use_label_encoder=False, eval_metric='mlogloss'))
]

stacking_model = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression(),
    cv=5
)

print("\nОценка Stacking модели:")
stacking_metrics = evaluate_model(stacking_model, X_train, y_train)
print(f"F1: {stacking_metrics['F1']:.3f} | Precision: {stacking_metrics['Precision']:.3f} | Recall: {stacking_metrics['Recall']:.3f}")

# Обучение и оценка Stacking
stacking_model.fit(X_train, y_train)
y_val_stacking = stacking_model.predict(X_val)
print("\nStacking модель:")
print(classification_report(y_val, y_val_stacking, target_names=label_encoder.classes_))

In [18]:
len(y_encoded)

64536

In [25]:
len(X)

64536

In [11]:
from tensorflow.keras.models import Sequential
import tensorflow as tf
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from tensorflow.keras.metrics import Precision, Recall, AUC
half_size = len(X_enc_log_std_norm) // 20
X = X_enc_log_std_norm.iloc[:half_size]
y = y_enc.iloc[:half_size]
y = y['State_FAILED']
seq_length = 10
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
model = Sequential([
    LSTM(64, input_shape=(seq_length, 1), return_sequences=True),
    Dropout(0.2),
    LSTM(32),
    Dense(1, activation='sigmoid')  # Бинарная классификация
])

def focal_loss(gamma=2.0, alpha=0.25):
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        pt = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
        loss = -tf.reduce_mean(alpha * tf.pow(1 - pt, gamma) * tf.math.log(pt))
        return loss
    return loss

model.compile(
    optimizer='adam',
    loss=focal_loss(),  # Вместо binary_crossentropy
    metrics=['Precision', 'Recall', 'AUC', tf.keras.metrics.AUC(curve='PR', name='auc_pr')]
)

# 4. Обучение с учетом дисбаланса (class_weight)
class_weight = {0: 1, 1: 20}  # Веса для классов
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.1,
    class_weight=class_weight
)

# 5. Оценка
y_pred = (model.predict(X_test) > 0.5).astype(int)
print(classification_report(y_test, y_pred, target_names=["Class 0", "Class 1"]))

E:\Programming\python\SC_anomaly_classifier\venv\lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/20
1453/1453 ━━━━━━━━━━━━━━━━━━━━ 210s 142ms/step - AUC: 0.5006 - Precision: 0.0000e+00 - Recall: 0.0000e+00 - auc_pr: 0.0307 - loss: 0.0219 - val_AUC: 0.5101 - val_Precision: 0.0000e+00 - val_Recall: 0.0000e+00 - val_auc_pr: 0.0426 - val_loss: 0.0099
Epoch 2/20
1453/1453 ━━━━━━━━━━━━━━━━━━━━ 203s 139ms/step - AUC: 0.5403 - Precision: 0.2327 - Recall: 0.0060 - auc_pr: 0.0515 - loss: 0.0199 - val_AUC: 0.5376 - val_Precision: 0.8333 - val_Recall: 0.0321 - val_auc_pr: 0.0719 - val_loss: 0.0098
Epoch 3/20
1453/1453 ━━━━━━━━━━━━━━━━━━━━ 189s 130ms/step - AUC: 0.5748 - Precision: 0.2043 - Recall: 0.0052 - auc_pr: 0.0540 - loss: 0.0192 - val_AUC: 0.8692 - val_Precision: 0.0000e+00 - val_Recall: 0.0000e+00 - val_auc_pr: 0.1029 - val_loss: 0.0076
Epoch 4/20
1453/1453 ━━━━━━━━━━━━━━━━━━━━ 113s 78ms/step - AUC: 0.7903 - Precision: 0.0000e+00 - Recall: 0.0000e+00 - auc_pr: 0.0882 - loss: 0.0168 - val_AUC: 0.8835 - val_Precision: 0.0000e+00 - val_Recall: 0.0000e+00 - val_auc_pr: 0.1356 - va

KeyboardInterrupt: 

Для LSTM с loss='binary_crossentropy' метрики низкие. Требуется дополнительная предобработка данных и подбор параметров, но скорее всего ничего не выйдет и градиентный бустинг является наиболее эффективной моделью в нашей задаче